In [1]:

# STEP 1: GPU CHECK
# Explanation: Check whether Google Colab has an NVIDIA GPU.

import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

!nvidia-smi


TensorFlow Version: 2.21.0
GPU Available: []
/bin/bash: line 1: nvidia-smi: command not found


In [2]:

# STEP 2: Install libraries
# Explanation: Install FastAPI, ONNX and image processing tools.

!pip install -q fastapi uvicorn python-multipart pillow \
    tf2onnx onnx requests

print("Libraries installed successfully!")


Libraries installed successfully!


In [3]:

# STEP 3: Upload trained VisionPort model
# Explanation: Upload visionport_model.keras from Week 1.

from google.colab import files

uploaded = files.upload()

print("Model uploaded!")


Saving visionport_model.keras to visionport_model.keras
Model uploaded!


In [4]:

# STEP 4: Load AI model
# Explanation: Load the trained CNN model into memory.

import tensorflow as tf

model = tf.keras.models.load_model(
    "visionport_model.keras"
)

model.summary()

print("VisionPort model loaded successfully!")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,070,432 (4.08 MB)

 Trainable params: 356,810 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 713,622 (2.72 MB)

VisionPort model loaded successfully!


In [5]:

# STEP 5: GPU Inference
# Explanation: Run a sample prediction using the available GPU.

import numpy as np

print("Available GPUs:",
      tf.config.list_physical_devices("GPU"))

dummy_image = np.random.rand(
    1, 32, 32, 3
).astype("float32")

prediction = model.predict(dummy_image, verbose=0)

print("Prediction output:", prediction)
print("Predicted class:", np.argmax(prediction))
print("GPU inference test completed!")


Available GPUs: []
Prediction output: [[0.02298071 0.11666988 0.00459648 0.0040394  0.00134042 0.00135374
  0.09421902 0.0038911  0.00544996 0.7454593 ]]
Predicted class: 9
GPU inference test completed!


In [6]:

# STEP 6: Convert TensorFlow model into ONNX
# Explanation: Convert the VisionPort CNN model into ONNX
# using a TensorFlow function to avoid Keras conversion issues.

import tensorflow as tf
import tf2onnx
import onnx

# Define input shape
input_signature = [
    tf.TensorSpec(
        shape=(1, 32, 32, 3),
        dtype=tf.float32,
        name="input"
    )
]

# Create TensorFlow inference function
@tf.function(input_signature=input_signature)
def visionport_function(x):
    return {"output": model(x, training=False)}

# Convert model to ONNX
onnx_model, _ = tf2onnx.convert.from_function(
    visionport_function,
    input_signature=input_signature,
    opset=17,
    output_path="visionport.onnx"
)

# Verify ONNX model
onnx.checker.check_model(onnx_model)

print("ONNX model exported successfully!")
print("ONNX model verified successfully!")


ONNX model exported successfully!
ONNX model verified successfully!


In [7]:

# STEP 7: Triton Model Repository
# Explanation: Prepare the directory structure expected
# by NVIDIA Triton Inference Server.

import os
import shutil

os.makedirs(
    "model_repository/visionport/1",
    exist_ok=True
)

shutil.copy(
    "visionport.onnx",
    "model_repository/visionport/1/model.onnx"
)

config = """
name: "visionport"
platform: "onnxruntime_onnx"
max_batch_size: 0

input [
  {
    name: "input"
    data_type: TYPE_FP32
    dims: [ 1, 32, 32, 3 ]
  }
]

output [
  {
    name: "output_0"
    data_type: TYPE_FP32
    dims: [ 1, 10 ]
  }
]
"""

with open(
    "model_repository/visionport/config.pbtxt",
    "w"
) as f:
    f.write(config)

print("Triton model repository prepared!")


Triton model repository prepared!


In [8]:

# STEP 8: FastAPI Backend
# Explanation: Create an API that accepts an image
# and sends it to the trained VisionPort model.

%%writefile app.py

from fastapi import FastAPI, UploadFile, File, HTTPException
from PIL import Image
import tensorflow as tf
import numpy as np
import io

app = FastAPI(title="VisionPort API")

model = tf.keras.models.load_model(
    "visionport_model.keras"
)

class_names = [
    "Airplane", "Automobile", "Bird", "Cat", "Deer",
    "Dog", "Frog", "Horse", "Ship", "Truck"
]

@app.get("/")
def home():
    return {
        "project": "VisionPort",
        "status": "Running"
    }

@app.get("/health")
def health():
    return {
        "api": "healthy",
        "gpu": str(tf.config.list_physical_devices("GPU"))
    }

@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(400, "Upload an image file")

    try:
        contents = await file.read()

        image = Image.open(
            io.BytesIO(contents)
        ).convert("RGB")

        image = image.resize((32, 32))

        image_array = np.array(
            image, dtype=np.float32
        ) / 255.0

        image_array = np.expand_dims(
            image_array, axis=0
        )

        result = model.predict(
            image_array, verbose=0
        )

        index = int(np.argmax(result[0]))

        return {
            "project": "VisionPort",
            "prediction": class_names[index],
            "confidence": float(result[0][index]),
            "status": "success"
        }

    except Exception as e:
        raise HTTPException(500, str(e))


Writing app.py


In [9]:

# STEP 9: Start FastAPI
# Explanation: Start the VisionPort backend inside Colab.

import subprocess
import time

server = subprocess.Popen([
    "uvicorn",
    "app:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

time.sleep(5)

print("VisionPort FastAPI server started!")


VisionPort FastAPI server started!


In [10]:

# STEP 10: API Health Check
# Explanation: Check whether the FastAPI application is running.

import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print("Status Code:", response.status_code)
print("Response:", response.json())


Status Code: 200
Response: {'api': 'healthy', 'gpu': '[]'}


In [11]:

# STEP 11: Test Prediction
# Explanation: Upload a sample image to the FastAPI endpoint.

from PIL import Image
import numpy as np
import requests

# Create a sample test image
sample = np.random.randint(
    0, 256, (32, 32, 3), dtype=np.uint8
)

Image.fromarray(sample).save("sample.jpg")

with open("sample.jpg", "rb") as image_file:

    response = requests.post(
        "http://127.0.0.1:8000/predict",
        files={
            "file": (
                "sample.jpg",
                image_file,
                "image/jpeg"
            )
        }
    )

print("Status:", response.status_code)
print("Prediction:", response.json())


Status: 200
Prediction: {'project': 'VisionPort', 'prediction': 'Frog', 'confidence': 0.6859895586967468, 'status': 'success'}


In [12]:

# STEP 12: Save Deployment Files
# Explanation: Package model, ONNX export and
# Triton repository for future deployment.

import shutil

shutil.make_archive(
    "VisionPort_Week2",
    "zip",
    ".",
    "model_repository"
)

print("VisionPort deployment files prepared!")


VisionPort deployment files prepared!


In [13]:

# STEP 13: TensorRT Check
# Explanation: Check whether NVIDIA TensorRT is available
# in the current Colab environment.

try:
    import tensorrt as trt

    print("TensorRT Version:", trt.__version__)
    print("TensorRT is available!")

except ImportError:
    print("TensorRT is not installed in this Colab runtime.")
    print("ONNX model is ready for later TensorRT conversion.")


TensorRT is not installed in this Colab runtime.
ONNX model is ready for later TensorRT conversion.


In [14]:

# STEP 14: Download project files
# Explanation: Download the model repository and ONNX model.

from google.colab import files

files.download("visionport.onnx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>